# Results Dashboard

This notebook is a reporting and parsing notebook only. It does not run the heavy experiments itself.

It reads JSON outputs produced by:
- `cluster/run_geometry_benchmark.py`
- `cluster/run_angle_map.py`
- `cluster/run_supervised_grid.py`

The defaults below prefer full-result files first. If a full JSON is not available yet, the notebook falls back to the smoke file for that section.

In [ ]:
import sys
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, '../src')

from rp_study.experiments.supervised_training import EpochMetrics, SupervisedTrainingResult
from rp_study.visualization.training_plots import plot_supervised_comparison_matrix, plot_training_histories

plt.style.use('default')

## Configure Result Files

These defaults prefer full outputs if they exist locally. Otherwise they fall back to the smoke-result filenames.

In [ ]:
RESULTS_DIR = Path('../reports/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def first_existing(*names):
    for name in names:
        path = RESULTS_DIR / name
        if path.exists():
            return path
    return RESULTS_DIR / names[0]


def first_existing_or_none(*names):
    for name in names:
        path = RESULTS_DIR / name
        if path.exists():
            return path
    return None


GEOMETRY_PATH = first_existing('geometry_benchmark_cifar10.json', 'geometry_smoke.json')
ANGLE_PATH = first_existing('angle_map.json', 'angle_smoke.json')
SUPERVISED_PATH = first_existing_or_none('supervised_comparison.json', 'supervised_smoke.json')

print('Results dir:       ', RESULTS_DIR.resolve())
print('Geometry path:     ', GEOMETRY_PATH, GEOMETRY_PATH.exists())
print('Angle-map path:    ', ANGLE_PATH, ANGLE_PATH.exists())
print('Supervised path:   ', SUPERVISED_PATH, SUPERVISED_PATH.exists() if SUPERVISED_PATH else False)

print('\nAvailable JSON files:')
json_paths = sorted(RESULTS_DIR.glob('*.json'))
if json_paths:
    for path in json_paths:
        print(' -', path.name)
else:
    print(' - none yet; copy results from the cluster into reports/results/')

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text())


def load_supervised_results(path):
    raw = load_json(path)
    results = []
    for item in raw:
        history = [EpochMetrics(**epoch) for epoch in item.get('history', [])]
        results.append(
            SupervisedTrainingResult(
                classifier_config=item['classifier_config'],
                training_config=item['training_config'],
                status=item['status'],
                best_epoch=item['best_epoch'],
                best_test_accuracy=item['best_test_accuracy'],
                final_train_accuracy=item['final_train_accuracy'],
                final_test_accuracy=item['final_test_accuracy'],
                history=history,
                parameter_count=item['parameter_count'],
            )
        )
    return results


def print_geometry_table(rows, metric_name, value_fmt='>.3f'):
    init_strategies = sorted({row['init_strategy'] for row in rows})
    depths = sorted({row['depth'] for row in rows})
    print(f'\n--- {metric_name} ---')
    print(f"{'Initializer':>32s}", end='')
    for depth in depths:
        print(f'  {depth:>4d}L', end='')
    print()
    print('-' * (32 + 7 * len(depths)))
    for init_strategy in init_strategies:
        print(f'{init_strategy:>32s}', end='')
        strategy_rows = {row['depth']: row for row in rows if row['init_strategy'] == init_strategy}
        for depth in depths:
            value = strategy_rows[depth][metric_name]
            if isinstance(value, (int, float)) and np.isfinite(value):
                print(f'  {value:{value_fmt}}', end='')
            else:
                print('    nan', end='')
        print()


def plot_geometry_rows(rows, title='Geometry Preservation vs Depth'):
    metrics = [
        ('knn_accuracy', 'k-NN Accuracy'),
        ('distance_correlation', 'Pairwise Distance Correlation'),
        ('effective_dim', 'Effective Dimensionality'),
    ]
    init_strategies = sorted({row['init_strategy'] for row in rows})
    depths = sorted({row['depth'] for row in rows})
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (metric_name, metric_label) in zip(axes, metrics):
        for init_strategy in init_strategies:
            strategy_rows = sorted(
                [row for row in rows if row['init_strategy'] == init_strategy],
                key=lambda row: row['depth'],
            )
            depths_local = [row['depth'] for row in strategy_rows]
            values = [row[metric_name] for row in strategy_rows]
            ax.plot(depths_local, values, 'o-', label=init_strategy, markersize=5)
        ax.set_xlabel('Depth')
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.set_xticks(depths)
        ax.grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    return fig


def plot_angle_map_payload(payload, title='Empirical Angle Map'):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    left_ax, right_ax = axes
    colors = ['tab:blue', 'tab:green', 'tab:red', 'tab:orange', 'tab:purple', 'tab:brown']
    pi_ticks = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi]
    pi_labels = [r'$0$', r'$\frac{\pi}{4}$', r'$\frac{\pi}{2}$', r'$\frac{3\pi}{4}$', r'$\pi$']

    left_ax.plot([0, np.pi], [0, np.pi], 'k--', lw=2, label='Identity')
    right_ax.axhline(1.0, color='k', ls='--', lw=2, label='No contraction')

    for color, (init_strategy, item) in zip(colors, payload.items()):
        alpha_grid = np.array(item['alpha_grid'], dtype=float)
        output_mean = np.array(item['output_mean'], dtype=float)
        output_std = np.array(item['output_std'], dtype=float)
        ratio = np.divide(output_mean, alpha_grid, out=np.full_like(output_mean, np.nan), where=alpha_grid > 0)
        ratio_std = np.divide(output_std, alpha_grid, out=np.full_like(output_std, np.nan), where=alpha_grid > 0)

        left_ax.plot(alpha_grid, output_mean, '-o', color=color, markersize=3, lw=1.5, label=init_strategy)
        left_ax.fill_between(alpha_grid, output_mean - output_std, output_mean + output_std, color=color, alpha=0.15)

        right_ax.plot(alpha_grid, ratio, '-o', color=color, markersize=3, lw=1.5, label=init_strategy)
        right_ax.fill_between(alpha_grid, ratio - ratio_std, ratio + ratio_std, color=color, alpha=0.15)

    left_ax.set_xticks(pi_ticks)
    left_ax.set_xticklabels(pi_labels)
    left_ax.set_yticks(pi_ticks)
    left_ax.set_yticklabels(pi_labels)
    left_ax.set_xlabel(r'Input angle $\alpha$')
    left_ax.set_ylabel(r'Output angle $\alpha^\prime$')
    left_ax.set_title(r'$\alpha \to \alpha^\prime$')
    left_ax.set_xlim(0, np.pi)
    left_ax.set_ylim(0, np.pi)
    left_ax.set_aspect('equal')
    left_ax.grid(True, alpha=0.3)
    left_ax.legend(fontsize=8)

    right_ax.set_xticks(pi_ticks)
    right_ax.set_xticklabels(pi_labels)
    right_ax.set_xlabel(r'Input angle $\alpha$')
    right_ax.set_ylabel(r'Contraction ratio $\alpha^\prime / \alpha$')
    right_ax.set_title('Contraction Ratio')
    right_ax.set_ylim(0, 1.2)
    right_ax.grid(True, alpha=0.3)
    right_ax.legend(fontsize=8)

    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    return fig


def supervised_summary_rows(results):
    rows = []
    for result in results:
        rows.append(
            {
                'dataset': result.training_config['dataset'],
                'architecture': result.classifier_config['architecture'],
                'depth': result.classifier_config['depth'],
                'init_strategy': result.classifier_config['init_strategy'],
                'use_batch_norm': result.classifier_config['use_batch_norm'],
                'status': result.status,
                'best_epoch': result.best_epoch,
                'best_test_accuracy': result.best_test_accuracy,
                'final_test_accuracy': result.final_test_accuracy,
                'parameter_count': result.parameter_count,
            }
        )
    return rows

## Geometry Benchmark Results

In [ ]:
if GEOMETRY_PATH.exists():
    geometry_rows = load_json(GEOMETRY_PATH)
    print(f'Loaded {len(geometry_rows)} geometry rows from {GEOMETRY_PATH.name}')
    print_geometry_table(geometry_rows, 'knn_accuracy', value_fmt='>.3f')
    print_geometry_table(geometry_rows, 'distance_correlation', value_fmt='>.3f')
    print_geometry_table(geometry_rows, 'effective_dim', value_fmt='>5.1f')
    plot_geometry_rows(geometry_rows, title=f'Geometry Preservation: {GEOMETRY_PATH.stem}')
    plt.show()
else:
    geometry_rows = []
    print('Geometry JSON not found yet:', GEOMETRY_PATH)

## Angle-Map / Section 4 Results

In [ ]:
if ANGLE_PATH.exists():
    angle_payload = load_json(ANGLE_PATH)
    print(f'Loaded angle-map results for {len(angle_payload)} initializers from {ANGLE_PATH.name}')
    for init_strategy, item in angle_payload.items():
        alpha_grid = np.array(item['alpha_grid'], dtype=float)
        output_mean = np.array(item['output_mean'], dtype=float)
        ratio = np.divide(output_mean, alpha_grid, out=np.full_like(output_mean, np.nan), where=alpha_grid > 0)
        print(f'{init_strategy:>30s}  mean contraction ratio = {np.nanmean(ratio):.4f}')
    plot_angle_map_payload(angle_payload, title=f'Angle Map: {ANGLE_PATH.stem}')
    plt.show()
else:
    angle_payload = {}
    print('Angle-map JSON not found yet:', ANGLE_PATH)

## Supervised Comparison Results

In [ ]:
if SUPERVISED_PATH is not None and SUPERVISED_PATH.exists():
    supervised_results = load_supervised_results(SUPERVISED_PATH)
    supervised_rows = supervised_summary_rows(supervised_results)
    print(f'Loaded {len(supervised_results)} supervised runs from {SUPERVISED_PATH.name}')
    print('Datasets:     ', sorted({row['dataset'] for row in supervised_rows}))
    print('Architectures:', sorted({row['architecture'] for row in supervised_rows}))
    print('Depths:       ', sorted({row['depth'] for row in supervised_rows}))
    print('Initializers: ', sorted({row['init_strategy'] for row in supervised_rows}))
    print('\nTop runs by best test accuracy:')
    for row in sorted(supervised_rows, key=lambda item: item['best_test_accuracy'], reverse=True)[:12]:
        bn_label = 'BN' if row['use_batch_norm'] else 'No BN'
        print(
            f"{row['dataset']:>14s}  {row['architecture']:>3s}  {row['depth']:>3d}L  "
            f"{row['init_strategy']:>18s}  {bn_label:>5s}  "
            f"status={row['status']:>9s}  best={row['best_test_accuracy']:.4f}"
        )
else:
    supervised_results = []
    supervised_rows = []
    print('No supervised JSON found yet. Expected one of: supervised_smoke.json, supervised_comparison.json')

In [ ]:
if supervised_results:
    datasets = sorted({result.training_config['dataset'] for result in supervised_results})
    architectures = sorted({result.classifier_config['architecture'] for result in supervised_results})
    for dataset in datasets:
        for architecture in architectures:
            plot_supervised_comparison_matrix(
                supervised_results,
                dataset=dataset,
                architecture=architecture,
                metric='best_test_accuracy',
                figsize=(8, 5),
            )
            plt.show()
else:
    print('Skipping supervised comparison matrices until a supervised JSON file exists.')

## Training-History Drilldown

These defaults adapt to whichever supervised JSON is currently available.

In [ ]:
if SUPERVISED_PATH is not None and SUPERVISED_PATH.name == 'supervised_comparison.json':
    FILTER_DATASET = 'cifar10'
    FILTER_ARCHITECTURE = 'cnn'
    FILTER_DEPTH = 50
else:
    FILTER_DATASET = 'fashion_mnist'
    FILTER_ARCHITECTURE = 'fc'
    FILTER_DEPTH = 50

filtered_results = [
    result
    for result in supervised_results
    if result.training_config['dataset'] == FILTER_DATASET
    and result.classifier_config['architecture'] == FILTER_ARCHITECTURE
    and result.classifier_config['depth'] == FILTER_DEPTH
]

print(f'Filtered runs: {len(filtered_results)}')
for result in filtered_results:
    cfg = result.classifier_config
    bn_label = 'BN' if cfg['use_batch_norm'] else 'No BN'
    print(f" - {cfg['init_strategy']} / {bn_label}: best={result.best_test_accuracy:.4f}")

if filtered_results:
    plot_training_histories(filtered_results, metric='test_accuracy', figsize=(10, 5))
    plt.show()
    plot_training_histories(filtered_results, metric='train_accuracy', figsize=(10, 5))
    plt.show()
else:
    print('No runs match the current filters yet.')

## Recommended Workflow

1. Copy finished JSON outputs from the cluster into `reports/results/`.
2. Run the notebook once with the smoke files to verify parsing and plotting.
3. When the full runs finish, the same notebook will automatically prefer those files only if the smoke files are absent. You can also set the three path variables manually.
4. Start by inspecting geometry and angle-map results, then move to the supervised matrices and training curves.